# Point-source dose rates

Turn an activity into a dose rate: air kerma in Gy/h, ambient dose
equivalent in Sv/h, and the inverse-square law that governs them both.

**Goals**

1. Compute air-kerma and ambient dose rates for a point source.
2. Understand where the exposure-rate constant comes from.
3. Use the inverse-square law to move the detector.
4. Know the limits of the point-source model.

**Sources**

* ICRP Publication 74 (H*(10)/Ka conversion, Table A.21):
  https://www.icrp.org/publication.asp?id=ICRP+Publication+74
* Lauridsen, B. (1982), "Table of Exposure Rate Constants and Dose
  Equivalent Rate Constants", Risoe-M-2322 (gamma constants, bundled).
* NIST Special Publication 811 (units): https://www.nist.gov/pml/special-publication-811


## From activity to dose rate

The bundled coefficient $\gamma$ (R.cm2.mCi-1.h-1) gives the exposure rate
for a point source at distance $r$:

$$\dot{X}[\mathrm{R/h}] = \gamma \cdot \frac{A[\mathrm{Bq}]}{3.7\times 10^7} \cdot \frac{1}{r_{cm}^2}$$

pydecay then converts with exact factors: `1 R in air = 8.76e-3 Gy`
(air kerma), and `quantity="ambient"` multiplies by the bundled ICRP-74
`H*(10)/Ka = 1.16 Sv/Gy` at 1.25 MeV (ICRP Publication 74 Table A.21,
https://www.icrp.org/publication.asp?id=ICRP+Publication+74).


In [ ]:
import math

from pydecay import dose_rate

a0 = 1e6  # 1 MBq of Co-60

kerma_1m = dose_rate(a0, "Co-60", 1.0)                       # Gy/h, default
ambient_1m = dose_rate(a0, "Co-60", 1.0, quantity="ambient")  # Sv/h
print(f"air kerma rate at 1 m:   {kerma_1m:.6e} Gy/h")
print(f"ambient dose equiv at 1m: {ambient_1m:.6e} Sv/h")
print(f"H*(10)/Ka implied:       {ambient_1m / kerma_1m:.4f}")


## The inverse-square law

Doubling the distance quarters the dose rate - exact for a point source,
before any air attenuation. Pass `attenuate_in_air=True` to multiply by
`exp(-mu_air * r)` at 1.25 MeV, a free-in-air refinement that matters at
large distances. Real sources are finite, so treat close-range results as
an upper bound on the gradient.


In [ ]:
kerma_2m = dose_rate(a0, "Co-60", 2.0)
print(f"ratio D(1 m) / D(2 m) = {kerma_1m / kerma_2m:.6f}   (expect exactly 4)")

kerma_1m_air = dose_rate(a0, "Co-60", 1.0, attenuate_in_air=True)
print(f"with air attenuation at 1 m: {kerma_1m_air:.6e} Gy/h "
      f"({100 * (1 - kerma_1m_air / kerma_1m):.4f}% lower)")


## Exercise: halve the dose rate

You need the dose rate to drop to **half** its value at 1 m. At what
distance should you move? Use the inverse-square law (no air attenuation)
and verify with `dose_rate`.

The solution is in the next cell, tagged `answer`.


In [ ]:
# ANSWER (cell tagged "answer" - hide while attempting the exercise)
r_half = 1.0 * math.sqrt(2)
d_here = dose_rate(a0, "Co-60", r_half)
print(f"at r = sqrt(2) m = {r_half:.6f} m: {d_here:.6e} Gy/h")
assert math.isclose(d_here, kerma_1m / 2, rel_tol=1e-12)
print("exercise checks passed")


## Golden check

Pinned in `tests/golden/golden_values.json`: the absolute air-kerma and
ambient rates for 1 MBq Co-60 at 1 m (analytic exposure formula with the
bundled Risoe-M-2322 gamma constant and the ICRP-74 H*(10)/Ka factor), plus
the exact inverse-square identity. `tests/test_notebooks.py` re-checks
every value printed below against that file.


In [ ]:
import json
import math

kerma = dose_rate(1e6, "Co-60", 1.0)
ambient = dose_rate(1e6, "Co-60", 1.0, quantity="ambient")
ratio = dose_rate(1e6, "Co-60", 1.0) / dose_rate(1e6, "Co-60", 2.0)

assert math.isclose(kerma, 3.0747600000000005e-07, rel_tol=1e-12)
assert math.isclose(ambient, 3.5667216000000003e-07, rel_tol=1e-12)
assert math.isclose(ratio, 4.0, rel_tol=1e-12)

payload = {
    "co60_kerma_1mbq_1m": kerma,
    "co60_ambient_1mbq_1m": ambient,
    "dose_inverse_square_identity": ratio,
}
print("GOLDEN_CHECK " + json.dumps(payload))
